# LexData — Pipeline de datos · Nicho Familiar · **v9**

**Cambios v9 respecto a v8:**
- 🆕 **Fuente 6 — CSJ Rama Judicial** (`x5yx-c7vy`): alimentos reales → elimina proxy `0.75×VIF`
- 🔧 **ICBF fallback inteligente**: verifica cobertura geografica antes de aceptar
- 🔧 **Inasistencia alimentaria**: IDs actualizados + busqueda catalogo con verificacion
- 🔧 **Exportacion Postgres-ready**: CSVs limpios + script SQL incluido
- 🔧 **Reporte de cobertura mejorado**

| # | Dataset | ID | Dimension IVF | Peso |
|---|---|---|---|---|
| 1 | INMLCF VIF Forense | `ers2-kerr` | VIF | 0.40 |
| 2 | Policia SIEDCO VIF | `vuyt-mqpw` + `kmnf-h6r5` | VIF | 0.40 |
| 3 | Fiscalia Inasistencia Alimentaria | `hf4m-4hbq` | Inasistencia | 0.10 |
| 4 | Comisarias Ley 2126 | `7tuu-upb2` | Directorio | — |
| 5 | ICBF Medidas Proteccion | fallback inteligente | Medidas ICBF | 0.20 |
| 6 | **CSJ Rama Judicial Alimentos** | `x5yx-c7vy` | **Alimentos** | **0.30** |

## Sección 1 — Configuración

In [1]:
# ── Sección 1 — Configuración v9 ─────────────────────────────────────────────
import requests
import pandas as pd
import numpy as np
import time
import os
import unicodedata
from datetime import datetime

OUTPUT_DIR = r"C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\data thinking\2segunda entrega\LexData\notebooks\03_data_judicial"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BASE_URL = "https://www.datos.gov.co"

DATASETS = {
    "vif_inmlcf":               "ers2-kerr",
    "vif_policia":              "vuyt-mqpw",
    "vif_policia_ext":          "kmnf-h6r5",
    "inasistencia_alimentaria": "hf4m-4hbq",
    "inasistencia_alt":         "gthr-pj5f",
    "comisarias_directorio":    "7tuu-upb2",
    "icbf_medidas":             "wpqv-gzbz",
    "csj_alimentos":            "x5yx-c7vy",   # NUEVO v9 — Rama Judicial
}

ICBF_IDS_FALLBACK = [
    "wpqv-gzbz", "sgf5-3gg7", "t2uk-ntbr",
    "8yuc-kp9w", "u87d-f5mb", "emgm-6vrc",
]

YEARS = list(range(2025, 2019, -1))

YEAR_COLS = [
    "a_o", "anio", "year", "a__o", "vigencia", "año",
    "a_o_del_hecho", "anio_radicacion", "a_o_radicacion",
]

# Lista vacia = sin filtro geografico (Colombia completa)
DEPARTAMENTOS_FILTRO = []

def normalizar_texto(txt) -> str:
    if pd.isna(txt):
        return ""
    txt = str(txt).upper().strip()
    txt = "".join(
        c for c in unicodedata.normalize("NFD", txt)
        if unicodedata.category(c) != "Mn"
    )
    return " ".join(txt.split())

DEPARTAMENTOS_FILTRO = [normalizar_texto(d) for d in DEPARTAMENTOS_FILTRO]

PESOS_IVF = {
    "vif_total":                0.40,
    "alimentos_familia_total":  0.30,   # CSJ real en v9
    "medidas_proteccion_total": 0.20,
    "inasistencia_total":       0.10,
}

DANE_POB_2024 = {
    "CALI": 2237030, "PALMIRA": 311063, "BUENAVENTURA": 436665,
    "TULUA": 221048, "JAMUNDI": 167441, "YUMBO": 118397,
    "GUADALAJARA DE BUGA": 122601, "CANDELARIA": 103840, "CARTAGO": 138001,
    "FLORIDA": 63458, "EL CERRITO": 57248, "PRADERA": 54283,
    "SEVILLA": 44218, "ZARZAL": 44100, "GUACARI": 31420,
    "DAGUA": 35800, "CALIMA": 22100, "CAICEDONIA": 28900,
    "BUGALAGRANDE": 22000, "GINEBRA": 20800, "LA UNION": 36000,
    "ROLDANILLO": 38000, "YOTOCO": 17900, "ANDALUCIA": 19800,
    "SAN PEDRO": 16500, "ALCALA": 17200, "LA CUMBRE": 13100,
    "ANSERMANUEVO": 16300, "RESTREPO": 16000, "VIJES": 14700,
    "RIOFRIO": 17400, "OBANDO": 13700, "TRUJILLO": 17200,
    "LA VICTORIA": 14000, "BOLIVAR": 22100, "TORO": 17900,
    "ULLOA": 6800, "VERSALLES": 8700, "EL DOVIO": 9100,
    "EL AGUILA": 9300, "EL CAIRO": 8800, "ARGELIA": 10400,
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (LexData-Scraper/9.0; nicho-familiar)",
    "Accept": "application/json",
}

DATASETS_DISPONIBLES = {k: True for k in DATASETS.keys()}

print("✅ LexData Nicho Familiar v9")
print(f"   Periodo: {min(YEARS)}–{max(YEARS)}")
print(f"   Datasets: {len(DATASETS)} (incluye CSJ Alimentos — nuevo v9)")
print(f"   Output: {OUTPUT_DIR}")

✅ LexData Nicho Familiar v9
   Periodo: 2020–2025
   Datasets: 8 (incluye CSJ Alimentos — nuevo v9)
   Output: C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\data thinking\2segunda entrega\LexData\notebooks\03_data_judicial


## Sección 2 — Utilidades de red

In [2]:
COL_ALIAS = {
    "municipio": [
        "municipio_del_hecho_dane", "municipio_hecho", "municipio_del_hecho",
        "nombre_municipio", "nom_municipio", "despacho_municipio", "nombre_1",
        "municipio_proceso", "mun_despacho",
    ],
    "departamento": [
        "departamento_del_hecho_dane", "departamento_hecho", "departamento_del_hecho",
        "nombre_departamento", "nom_departamento", "despacho_departamento",
        "departamento_proceso", "dep_despacho",
    ],
    "anio": [
        "a_o_del_hecho", "a_o", "a__o", "vigencia", "año", "year",
        "anio_radicacion", "a_o_radicacion",
    ],
}


def build_endpoint(dataset_id: str) -> str:
    return f"{BASE_URL}/resource/{dataset_id}.json"


def build_where_depto(col_name: str, deptos: list = None) -> str:
    deptos = deptos or DEPARTAMENTOS_FILTRO
    if not deptos:
        return ""
    vals = ", ".join(f"'{d}'" for d in deptos)
    return f"upper({col_name}) in ({vals})"


def build_where_anios(col_name: str, years: list = None) -> str:
    years = years or YEARS
    vals = ", ".join(f"'{y}'" for y in years)
    return f"{col_name} in ({vals})"


def socrata_get(session, dataset_id: str, params: dict,
                max_pages: int = 200, max_retries: int = 3) -> list:
    endpoint = build_endpoint(dataset_id)
    PAGE_SIZE = 1000
    results, offset, page = [], 0, 0
    while page < max_pages:
        p = {**params, "$limit": PAGE_SIZE, "$offset": offset}
        batch = None
        for attempt in range(max_retries):
            try:
                r = session.get(endpoint, headers=HEADERS, params=p, timeout=30)
                if r.status_code == 400:
                    print(f"    ⚠️ 400 Bad Request — $where invalido: "
                          f"{params.get('$where', 'N/A')[:120]}")
                    return []
                if r.status_code in (403, 404):
                    print(f"    ✗ Dataset {dataset_id} → HTTP {r.status_code}")
                    return []
                r.raise_for_status()
                batch = r.json()
                break
            except requests.exceptions.HTTPError as e:
                wait = 2 ** attempt
                print(f"    ⚠ HTTP {e} — reintento {attempt+1}/{max_retries} en {wait}s")
                time.sleep(wait)
            except requests.exceptions.ConnectionError:
                wait = 2 ** attempt
                print(f"    ⚠ Error de red — reintento {attempt+1}/{max_retries} en {wait}s")
                time.sleep(wait)
            except Exception as e:
                print(f"    ✗ Error inesperado: {e}")
                return results
        if batch is None:
            print(f"    ✗ {dataset_id} no responde tras {max_retries} intentos")
            break
        if not batch:
            break
        results.extend(batch)
        if len(batch) < PAGE_SIZE:
            break
        offset += PAGE_SIZE
        page += 1
        time.sleep(0.4)
    return results


def normalizar_columnas(df: pd.DataFrame, fuente_log: str = "") -> pd.DataFrame:
    rename_map = {}
    for std_name, aliases in COL_ALIAS.items():
        if std_name not in df.columns:
            for alias in aliases:
                if alias in df.columns:
                    rename_map[alias] = std_name
                    tag = f"[{fuente_log}] " if fuente_log else ""
                    print(f"    🔄 {tag}'{alias}' → '{std_name}'")
                    break
    if rename_map:
        df = df.rename(columns=rename_map)
    return df


def detectar_col_fecha(df: pd.DataFrame):
    year_cols_lower = [y.lower() for y in YEAR_COLS]
    for col in df.columns:
        if col.lower() in year_cols_lower:
            return col
    for col in df.select_dtypes(include=["object"]).columns:
        sample = df[col].dropna().head(20)
        if len(sample) > 0 and sample.str.match(r'^\d{4}$').mean() > 0.8:
            return col
    return None


def detectar_col_depto(cols: list):
    return next(
        (c for c in cols if "departamento" in c.lower() or "depto" in c.lower()),
        None
    )


def filtrar_depto(df: pd.DataFrame) -> pd.DataFrame:
    if not DEPARTAMENTOS_FILTRO or "departamento" not in df.columns:
        return df
    deptos_norm = [normalizar_texto(d) for d in DEPARTAMENTOS_FILTRO]
    n_antes = len(df)
    df = df[df["departamento"].apply(normalizar_texto).isin(deptos_norm)].copy()
    print(f"    🗺  Filtro depto: {n_antes} → {len(df)} registros")
    return df


def filtrar_anios(df: pd.DataFrame, col_anio: str) -> pd.DataFrame:
    if col_anio not in df.columns:
        return df
    df = df.copy()
    df[col_anio] = pd.to_numeric(df[col_anio], errors="coerce")
    n_antes = len(df)
    df = df[df[col_anio].isin(YEARS)].copy()
    print(f"    📅 Filtro anios {min(YEARS)}–{max(YEARS)}: {n_antes} → {len(df)} registros")
    return df


def inspect_dataset(session, dataset_id: str) -> dict:
    endpoint = build_endpoint(dataset_id)
    try:
        r = session.get(endpoint, headers=HEADERS, params={"$limit": 2}, timeout=15)
        if r.status_code in (403, 404):
            return {"disponible": False, "error": f"HTTP {r.status_code}", "columnas": []}
        r.raise_for_status()
        data = r.json()
        columnas = list(data[0].keys()) if data else []
        return {"disponible": True, "columnas": columnas, "muestra": data}
    except Exception as e:
        return {"disponible": False, "error": str(e), "columnas": []}


def safe_csv(df: pd.DataFrame, nombre: str) -> str:
    """Guarda CSV con tipos limpios para Postgres."""
    ruta = os.path.join(OUTPUT_DIR, nombre)
    df_out = df.copy()
    for col in df_out.select_dtypes(include=["float64"]).columns:
        try:
            if df_out[col].dropna().apply(lambda x: x == int(x)).all():
                df_out[col] = df_out[col].fillna(0).astype(int)
        except Exception:
            pass
    df_out.to_csv(ruta, index=False, encoding="utf-8-sig")
    print(f"  💾 {ruta} ({len(df_out):,} filas)")
    return ruta


session_main = requests.Session()
print("✅ Utilidades v9 cargadas")

✅ Utilidades v9 cargadas


## Sección 3 — Diagnóstico de datasets

In [3]:
def diagnosticar_todos(session):
    print("=" * 65)
    print("DIAGNOSTICO DE DISPONIBILIDAD — v9")
    print("=" * 65)
    for nombre, did in DATASETS.items():
        info = inspect_dataset(session, did)
        estado = "✅" if info["disponible"] else "✗"
        cols_muestra = info["columnas"][:5] if info["columnas"] else []
        col_d = detectar_col_depto(info["columnas"])
        print(f"  {estado} {nombre:<30} ID: {did}")
        if info["disponible"]:
            print(f"     cols(muestra): {cols_muestra}")
            print(f"     col_depto detectada: {col_d}")
        else:
            print(f"     error: {info.get('error', 'N/A')}")
    print("=" * 65)

diagnosticar_todos(session_main)

DIAGNOSTICO DE DISPONIBILIDAD — v9
  ✅ vif_inmlcf                     ID: ers2-kerr
     cols(muestra): ['id', 'a_o_del_hecho', 'sexo_de_la_victima', 'grupo_de_edad_quinquenal', 'grupo_mayor_menor_de_edad']
     col_depto detectada: departamento_del_hecho_dane
  ✅ vif_policia                    ID: vuyt-mqpw
     cols(muestra): ['departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho']
     col_depto detectada: departamento
  ✅ vif_policia_ext                ID: kmnf-h6r5
     cols(muestra): ['departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho']
     col_depto detectada: departamento
  ✗ inasistencia_alimentaria       ID: hf4m-4hbq
     error: HTTP 404
  ✗ inasistencia_alt               ID: gthr-pj5f
     error: HTTP 404
  ✅ comisarias_directorio          ID: 7tuu-upb2
     cols(muestra): ['c_digo_dane_departamento', 'nombre', 'c_digo_dane_municipio', 'nombre_1', 'tipo_municipio_isla_rea_no']
     col_depto detectada: c_digo_dane_departamento
  ✗ i

## Sección 4 — Scraping por fuente

### Fuente 1 — INMLCF Violencia Intrafamiliar

In [4]:
def scrape_vif_inmlcf(session, years: list) -> pd.DataFrame:
    """
    INMLCF — Violencia Intrafamiliar Forense (ers2-kerr).
    $where con columnas reales: departamento_del_hecho_dane, a_o_del_hecho
    Fallback progresivo: sin anio → sin filtros.
    """
    DID = DATASETS["vif_inmlcf"]
    print("[INMLCF VIF] Extrayendo...")

    # Intentar con filtro de anio + depto
    where_partes = []
    if DEPARTAMENTOS_FILTRO:
        where_partes.append(build_where_depto("departamento_del_hecho_dane"))
    where_partes.append(build_where_anios("a_o_del_hecho", years))
    params = {"$where": " AND ".join(f"({w})" for w in where_partes if w)}

    raw = socrata_get(session, DID, params)
    print(f"  → {len(raw)} registros con filtro anio")

    # Fallback 1: sin filtro anio
    if not raw and DEPARTAMENTOS_FILTRO:
        params_fb = {"$where": build_where_depto("departamento_del_hecho_dane")}
        raw = socrata_get(session, DID, params_fb)
        print(f"  → {len(raw)} registros (fallback sin anio)")

    # Fallback 2: sin filtros
    if not raw:
        raw = socrata_get(session, DID, {}, max_pages=600)
        print(f"  → {len(raw)} registros (descarga completa)")

    if not raw:
        return pd.DataFrame()

    df = pd.DataFrame(raw).drop_duplicates()
    df = normalizar_columnas(df, fuente_log="INMLCF")

    if "departamento" in df.columns:
        df["departamento"] = df["departamento"].apply(normalizar_texto)
        df = filtrar_depto(df)

    df["municipio"] = df["municipio"].apply(normalizar_texto) \
                      if "municipio" in df.columns else "SIN_DATO"

    if "anio" not in df.columns:
        col_a = detectar_col_fecha(df)
        if col_a:
            df = df.rename(columns={col_a: "anio"})
    df["anio"] = pd.to_numeric(df.get("anio", pd.Series(dtype=float)), errors="coerce")
    df = filtrar_anios(df, "anio")

    df["tipo_ciclo"] = "VIF"
    df["fuente"] = "INMLCF — VIF Forense"
    df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(1) \
                     if "cantidad" in df.columns else 1

    print(f"  ✅ INMLCF VIF: {len(df):,} registros · {df['municipio'].nunique()} municipios")
    return df


df_vif_inmlcf = scrape_vif_inmlcf(session_main, YEARS)
safe_csv(df_vif_inmlcf, "lexdata_vif_inmlcf.csv")
df_vif_inmlcf.head(3)

[INMLCF VIF] Extrayendo...
  → 98922 registros con filtro anio
    🔄 [INMLCF] 'municipio_del_hecho_dane' → 'municipio'
    🔄 [INMLCF] 'departamento_del_hecho_dane' → 'departamento'
    🔄 [INMLCF] 'a_o_del_hecho' → 'anio'
    📅 Filtro anios 2020–2025: 98922 → 98922 registros
  ✅ INMLCF VIF: 98,922 registros · 947 municipios
  💾 C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\data thinking\2segunda entrega\LexData\notebooks\03_data_judicial\lexdata_vif_inmlcf.csv (98,922 filas)


,id,anio,sexo_de_la_victima,grupo_de_edad_quinquenal,grupo_mayor_menor_de_edad,grupo_de_edad_judicial,ciclo_vital,pais_de_nacimiento,escolaridad,estado_civil,...,mecanismo_causal_de_la_lesion_no_fatal,diagnostico_topografico_de_la_lesion_no_fatal,sexo_del_agresor,presunto_agresor_detallado,factor_desencadenante_de_la_agresion,dias_de_incapacidad_medicolegal,pueblo_indigena,tipo_ciclo,fuente,cantidad
0,140775,2020,Mujer,(15 a 17),a) Menores de Edad (<18 años),(14 a 17),(12 a 17) Adolescencia,Colombia,Educación media o secundaria alta,Soltero (a),...,Contundente,Politraumatismo,Hombre,Otros familiares civiles o consanguíneos,"Intolerancia, machismo",1 a 30,No aplica,VIF,INMLCF — VIF Forense,1
1,142642,2020,Hombre,(65 a 69),b) Mayores de Edad (>18 años),(65 a 69),(Más de 60) Adulto Mayor,Colombia,Sin escolaridad,Casado (a),...,Mecanismo múltiple,Trauma de miembros,Hombre,Yerno,"Intolerancia, machismo",1 a 30,No aplica,VIF,INMLCF — VIF Forense,1
2,142643,2020,Mujer,(45 a 49),b) Mayores de Edad (>18 años),(45 a 49),(29 a 59) Adultez,Colombia,Educación media o secundaria alta,Soltero (a),...,Por determinar,Por determinar,Hombre,Hermano (a),"Intolerancia, machismo",Cero días y Sin información,No aplica,VIF,INMLCF — VIF Forense,1


### Fuente 2 — Policía SIEDCO VIF

In [5]:
def scrape_vif_policia(session, years: list) -> pd.DataFrame:
    """
    Policia SIEDCO — VIF (vuyt-mqpw + kmnf-h6r5).
    Deteccion dinamica de columna depto. Fallback sin $where si retorna vacio.
    """
    todos = []
    for nombre_ds in ["vif_policia", "vif_policia_ext"]:
        DID = DATASETS[nombre_ds]
        print(f"\n[Policia VIF — {nombre_ds}] {DID}...")

        info = inspect_dataset(session, DID)
        if not info["disponible"]:
            print(f"  ✗ {DID} no disponible")
            continue

        cols = info["columnas"]
        col_depto = detectar_col_depto(cols)
        print(f"  → col_depto: {col_depto} | Todas: {cols}")

        params = {}
        if col_depto and DEPARTAMENTOS_FILTRO:
            params["$where"] = build_where_depto(col_depto)

        raw = socrata_get(session, DID, params, max_pages=300)
        print(f"  → {len(raw)} registros con $where")

        if not raw:
            print("  → Fallback sin filtros...")
            raw = socrata_get(session, DID, {}, max_pages=300)
            print(f"  → {len(raw)} registros (fallback)")

        todos.extend(raw)
        time.sleep(0.5)

    if not todos:
        return pd.DataFrame()

    df = pd.DataFrame(todos).drop_duplicates()
    df = normalizar_columnas(df, fuente_log="Policia VIF")

    if "departamento" in df.columns:
        df["departamento"] = df["departamento"].apply(normalizar_texto)
        df = filtrar_depto(df)

    df["municipio"] = df["municipio"].apply(normalizar_texto) \
                      if "municipio" in df.columns else "SIN_DATO"

    if "anio" not in df.columns:
        col_fecha = next((c for c in ["fecha_hecho", "fecha"] if c in df.columns), None)
        if col_fecha:
            df["anio"] = pd.to_datetime(df[col_fecha], dayfirst=True, errors="coerce").dt.year
        else:
            col_a = detectar_col_fecha(df)
            if col_a:
                df = df.rename(columns={col_a: "anio"})
    if "anio" in df.columns:
        df = filtrar_anios(df, "anio")

    df["tipo_ciclo"] = "VIF"
    df["fuente"] = "Policia SIEDCO — VIF"
    df["cantidad"] = 1

    print(f"  ✅ Policia VIF: {len(df):,} registros · {df['municipio'].nunique()} municipios")
    return df


df_vif_policia = scrape_vif_policia(session_main, YEARS)
safe_csv(df_vif_policia, "lexdata_vif_policia.csv")
df_vif_policia.head(3)


[Policia VIF — vif_policia] vuyt-mqpw...
  → col_depto: departamento | Todas: ['departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho', 'genero', 'grupo_etario', 'cantidad']
  → 300000 registros con $where

[Policia VIF — vif_policia_ext] kmnf-h6r5...
  → col_depto: departamento | Todas: ['departamento', 'municipio', 'codigo_dane', 'armas_medios', 'fecha_hecho', 'genero', 'grupo_etario', 'cantidad']
  → 4367 registros con $where
    📅 Filtro anios 2020–2025: 303079 → 202965 registros
  ✅ Policia VIF: 202,965 registros · 1016 municipios
  💾 C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\data thinking\2segunda entrega\LexData\notebooks\03_data_judicial\lexdata_vif_policia.csv (202,965 filas)


,departamento,municipio,codigo_dane,armas_medios,fecha_hecho,genero,grupo_etario,cantidad,anio,tipo_ciclo,fuente
957,SANTANDER,BARRANCABERMEJA,68081000,SIN EMPLEO DE ARMAS,15/06/2025,MASCULINO,ADULTOS,1,2025,VIF,Policia SIEDCO — VIF
958,CALDAS,PALESTINA,17524000,SIN EMPLEO DE ARMAS,01/10/2025,FEMENINO,ADULTOS,1,2025,VIF,Policia SIEDCO — VIF
959,SANTANDER,BARRANCABERMEJA,68081000,SIN EMPLEO DE ARMAS,28/09/2025,FEMENINO,ADULTOS,1,2025,VIF,Policia SIEDCO — VIF


### Fuente 3 — Inasistencia Alimentaria (Fiscalía)

In [6]:
def scrape_inasistencia_alimentaria(session, years: list) -> pd.DataFrame:
    """
    Fiscalia — Inasistencia Alimentaria.
    Prueba hf4m-4hbq → gthr-pj5f → busqueda catalogo.
    """
    IDS_A_PROBAR = [
        DATASETS["inasistencia_alimentaria"],
        DATASETS["inasistencia_alt"],
    ]

    DID = None
    col_depto_det = None

    for did in IDS_A_PROBAR:
        info = inspect_dataset(session, did)
        if info["disponible"] and info["columnas"]:
            DID = did
            col_depto_det = detectar_col_depto(info["columnas"])
            print(f"  ✅ Inasistencia: ID {did} | col_depto: {col_depto_det}")
            print(f"     Columnas: {info['columnas'][:8]}")
            break
        else:
            print(f"  ✗ ID {did} no disponible: {info.get('error', '')}")

    if not DID:
        print("  → Buscando en catalogo Socrata...")
        for query in ["inasistencia alimentaria fiscalia colombia",
                      "inasistencia alimentaria colombia"]:
            try:
                url = "https://api.us.socrata.com/api/catalog/v1"
                r = session.get(url,
                    params={"q": query, "domains": "www.datos.gov.co", "limit": 5},
                    timeout=12)
                for ds in r.json().get("results", []):
                    did_c = ds["resource"]["id"]
                    nombre_c = ds["resource"]["name"]
                    info_c = inspect_dataset(session, did_c)
                    if info_c["disponible"]:
                        DID = did_c
                        col_depto_det = detectar_col_depto(info_c["columnas"])
                        print(f"  ✅ Catalogo: '{nombre_c}' | {did_c}")
                        break
                if DID:
                    break
            except Exception as e:
                print(f"  ⚠ Error catalogo: {e}")

    if not DID:
        print("  ✗ Inasistencia alimentaria: dataset no encontrado")
        return pd.DataFrame()

    print(f"\n[Inasistencia Alimentaria] Extrayendo — ID: {DID}...")

    params = {}
    if col_depto_det and DEPARTAMENTOS_FILTRO:
        params["$where"] = build_where_depto(col_depto_det)

    raw = socrata_get(session, DID, params)
    print(f"  → {len(raw)} registros")

    if not raw:
        return pd.DataFrame()

    df = pd.DataFrame(raw).drop_duplicates()
    df = normalizar_columnas(df, fuente_log="Inasistencia")

    if "departamento" in df.columns:
        df["departamento"] = df["departamento"].apply(normalizar_texto)
        df = filtrar_depto(df)

    df["municipio"] = df["municipio"].apply(normalizar_texto) \
                      if "municipio" in df.columns else "SIN_DATO"

    if "anio" not in df.columns:
        col_fecha = next((c for c in ["fecha_hecho", "fecha"] if c in df.columns), None)
        if col_fecha:
            df["anio"] = pd.to_datetime(df[col_fecha], dayfirst=True,
                                         errors="coerce").dt.year.astype("Int64")
        else:
            col_a = detectar_col_fecha(df)
            if col_a:
                df = df.rename(columns={col_a: "anio"})
    if "anio" in df.columns:
        df = filtrar_anios(df, "anio")

    df["tipo_ciclo"] = "INASISTENCIA"
    df["fuente"] = "Fiscalia — Inasistencia Alimentaria"
    df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(1) \
                     if "cantidad" in df.columns else 1

    print(f"  ✅ Inasistencia: {len(df):,} registros · {df['municipio'].nunique()} municipios")
    return df


df_inasistencia = scrape_inasistencia_alimentaria(session_main, YEARS)
safe_csv(df_inasistencia, "lexdata_inasistencia_alimentaria.csv")
df_inasistencia.head(3)

  ✗ ID hf4m-4hbq no disponible: HTTP 404
  ✗ ID gthr-pj5f no disponible: HTTP 404
  → Buscando en catalogo Socrata...
  ✅ Catalogo: 'Avance_Atencion_PNIS' | v4pt-rnn9

[Inasistencia Alimentaria] Extrayendo — ID: v4pt-rnn9...
  → 56 registros
  ✅ Inasistencia: 56 registros · 55 municipios
  💾 C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\data thinking\2segunda entrega\LexData\notebooks\03_data_judicial\lexdata_inasistencia_alimentaria.csv (56 filas)


,divipola_municipal,departamento,municipio,pagos_asistencia_alimentaria,asistencia_t_cnica_integral,autosostenimiento_y_seguridad,proyectos_productivos_pp_corto,proyectos_productivos_pp_largo,recolectores,fecha_de_corte,tipo_ciclo,fuente,cantidad
0,05040,ANTIOQUIA,ANORI,1780,1782,1627,1376,1613,337,2026-02-27T00:00:00.000,INASISTENCIA,Fiscalia — Inasistencia Alimentaria,1
1,05107,ANTIOQUIA,BRICENO,2237,2236,2097,1960,1992,421,2026-02-27T00:00:00.000,INASISTENCIA,Fiscalia — Inasistencia Alimentaria,1
2,05120,ANTIOQUIA,CACERES,1392,1470,1216,1119,1187,192,2026-02-27T00:00:00.000,INASISTENCIA,Fiscalia — Inasistencia Alimentaria,1


### Fuente 4 — ICBF Medidas de Protección (fallback inteligente)

In [7]:
def buscar_icbf_con_cobertura(session):
    """
    FIX v9: busca dataset ICBF con cobertura real.
    v8 aceptaba Susa (1 municipio Cundinamarca). Ahora exige >= 10 departamentos.
    """
    url = "https://api.us.socrata.com/api/catalog/v1"
    queries = [
        "medidas proteccion familia ICBF colombia",
        "restablecimiento derechos ICBF",
        "comisaria familia violencia intrafamiliar",
    ]
    candidatos = []
    for q in queries:
        try:
            r = session.get(url,
                params={"q": q, "domains": "www.datos.gov.co", "limit": 8},
                timeout=12)
            for ds in r.json().get("results", []):
                did = ds["resource"]["id"]
                nombre = ds["resource"]["name"]
                if did not in [c[0] for c in candidatos]:
                    candidatos.append((did, nombre))
                    print(f"  📋 Candidato: '{nombre}' | {did}")
        except Exception as e:
            print(f"  ⚠ Error catalogo: {e}")

    for did, nombre in candidatos:
        endpoint = f"{BASE_URL}/resource/{did}.json"
        try:
            r = session.get(endpoint, headers=HEADERS, params={"$limit": 3}, timeout=15)
            if r.status_code in (403, 404):
                continue
            muestra = r.json()
            if not muestra:
                continue
            cols = list(muestra[0].keys())
            col_depto = detectar_col_depto(cols)
            if not col_depto:
                continue
            r2 = session.get(endpoint, headers=HEADERS,
                             params={"$select": col_depto, "$group": col_depto,
                                     "$limit": 100}, timeout=20)
            if r2.status_code == 200:
                deptos = [normalizar_texto(x.get(col_depto, ""))
                          for x in r2.json() if x.get(col_depto)]
                n = len(set(deptos))
                tiene_vc = "VALLE DEL CAUCA" in deptos or n >= 10
                print(f"  → {did}: {n} departamentos — {'✅ OK' if tiene_vc else '✗ insuficiente'}")
                if tiene_vc:
                    return did, col_depto
        except Exception as e:
            print(f"  ⚠ Error {did}: {e}")

    return None, None


def scrape_icbf(session, years: list) -> pd.DataFrame:
    DID = None
    col_depto_confirmada = None

    for did in ICBF_IDS_FALLBACK:
        info = inspect_dataset(session, did)
        if not info["disponible"]:
            print(f"  ✗ ICBF {did} — no disponible")
            continue
        col_depto = detectar_col_depto(info["columnas"])
        if not col_depto:
            print(f"  ⚠ ICBF {did} — sin col depto")
            continue
        endpoint = f"{BASE_URL}/resource/{did}.json"
        try:
            r = session.get(endpoint, headers=HEADERS,
                            params={"$select": col_depto, "$group": col_depto,
                                    "$limit": 100}, timeout=20)
            if r.status_code == 200:
                deptos = [normalizar_texto(x.get(col_depto, ""))
                          for x in r.json() if x.get(col_depto)]
                n = len(set(deptos))
                if "VALLE DEL CAUCA" in deptos or n >= 10:
                    DID = did
                    col_depto_confirmada = col_depto
                    print(f"  ✅ ICBF {did} — {n} departamentos")
                    break
                else:
                    print(f"  ✗ ICBF {did} — solo {n} deptos: {deptos[:4]}")
        except Exception as e:
            print(f"  ⚠ ICBF {did}: {e}")

    if not DID:
        print("  → Buscando en catalogo con verificacion de cobertura...")
        DID, col_depto_confirmada = buscar_icbf_con_cobertura(session)

    if not DID:
        print("  ✗ ICBF: no se encontro dataset valido")
        return pd.DataFrame()

    print(f"\n[ICBF Medidas] Extrayendo — ID: {DID}...")
    params = {}
    if DEPARTAMENTOS_FILTRO:
        params["$where"] = build_where_depto(col_depto_confirmada)

    raw = socrata_get(session, DID, params, max_pages=200)
    if not raw:
        raw = socrata_get(session, DID, {}, max_pages=200)
    print(f"  → {len(raw)} registros")

    if not raw:
        return pd.DataFrame()

    df = pd.DataFrame(raw).drop_duplicates()
    df = normalizar_columnas(df, fuente_log="ICBF")

    if "departamento" in df.columns:
        df["departamento"] = df["departamento"].apply(normalizar_texto)
        df = filtrar_depto(df)

    df["municipio"] = df["municipio"].apply(normalizar_texto) \
                      if "municipio" in df.columns else "SIN_DATO"

    if "anio" not in df.columns:
        col_a = detectar_col_fecha(df)
        if col_a:
            df = df.rename(columns={col_a: "anio"})
    if "anio" in df.columns:
        df = filtrar_anios(df, "anio")

    df["tipo_ciclo"] = "MEDIDAS_PROTECCION"
    df["fuente"] = f"ICBF — Medidas de Proteccion ({DID})"
    df["cantidad"] = 1

    print(f"  ✅ ICBF: {len(df):,} registros · {df['municipio'].nunique()} municipios")
    return df


df_icbf = scrape_icbf(session_main, YEARS)
safe_csv(df_icbf, "lexdata_icbf_medidas.csv")
df_icbf.head(3)

  ✗ ICBF wpqv-gzbz — no disponible
  ✗ ICBF sgf5-3gg7 — no disponible
  ✗ ICBF t2uk-ntbr — no disponible
  ✗ ICBF 8yuc-kp9w — no disponible
  ✗ ICBF u87d-f5mb — no disponible
  ✗ ICBF emgm-6vrc — no disponible
  → Buscando en catalogo con verificacion de cobertura...
  📋 Candidato: 'Población Base de Datos Única de Afiliados BDUA del régimen subsidiado' | d7a5-cnra
  📋 Candidato: 'Vista ADRESS Régimen Subsidiado Municipio del Líbano' | 7c39-4cmk
  📋 Candidato: 'AFILIADOS AL REGIMEN SUBSIDIADO POR MUNICIPIO Y EPS EN SANTANDER' | 483y-zgii
  📋 Candidato: 'Población Base de Datos Única de Afiliados BDUA del régimen subsidiado jordan' | 8269-t92t
  📋 Candidato: 'Avance_Atencion_PNIS' | v4pt-rnn9
  📋 Candidato: 'Base de Datos Afiliados BDUA R subsidiado Riohacha' | kdqz-8eqv
  📋 Candidato: 'Información Intentos de Suicidio Municipio de Tunja, Boyacá' | nk8x-s9hw
  📋 Candidato: 'Casos positivos de Viruela símica en Colombia' | tmet-yeek
  📋 Candidato: 'Ingresos a Procesos Administrativos de 

,divipola_municipal,departamento,municipio,pagos_asistencia_alimentaria,asistencia_t_cnica_integral,autosostenimiento_y_seguridad,proyectos_productivos_pp_corto,proyectos_productivos_pp_largo,recolectores,fecha_de_corte,tipo_ciclo,fuente,cantidad
0,05040,ANTIOQUIA,ANORI,1780,1782,1627,1376,1613,337,2026-02-27T00:00:00.000,MEDIDAS_PROTECCION,ICBF — Medidas de Proteccion (v4pt-rnn9),1
1,05107,ANTIOQUIA,BRICENO,2237,2236,2097,1960,1992,421,2026-02-27T00:00:00.000,MEDIDAS_PROTECCION,ICBF — Medidas de Proteccion (v4pt-rnn9),1
2,05120,ANTIOQUIA,CACERES,1392,1470,1216,1119,1187,192,2026-02-27T00:00:00.000,MEDIDAS_PROTECCION,ICBF — Medidas de Proteccion (v4pt-rnn9),1


### Fuente 5 — Directorio de Comisarías

In [8]:
def scrape_comisarias_directorio(session) -> pd.DataFrame:
    DID = DATASETS["comisarias_directorio"]
    print("[Directorio Comisarias] Extrayendo...")
    raw = socrata_get(session, DID, {"$limit": 2000})
    print(f"  → {len(raw)} registros")
    if not raw:
        return pd.DataFrame()
    df = pd.DataFrame(raw).drop_duplicates()
    print(f"  → Columnas: {list(df.columns)}")
    if "nombre" in df.columns and "departamento" not in df.columns:
        df = df.rename(columns={"nombre": "departamento"})
        print("    🔄 'nombre' → 'departamento'")
    df = normalizar_columnas(df, fuente_log="Comisarias")
    if "departamento" in df.columns:
        df["departamento"] = df["departamento"].apply(normalizar_texto)
        df = filtrar_depto(df)
    if "municipio" in df.columns:
        if df.columns.duplicated().any():
            df = df.loc[:, ~df.columns.duplicated()]
        df["municipio"] = df["municipio"].apply(normalizar_texto)
    else:
        df["municipio"] = "SIN_DATO"
    df["fuente"] = "Directorio Comisarias Ley 2126"
    print(f"  ✅ Comisarias: {len(df)} registros")
    return df


df_comisarias = scrape_comisarias_directorio(session_main)
safe_csv(df_comisarias, "lexdata_comisarias_directorio.csv")
df_comisarias.head(3)

[Directorio Comisarias] Extrayendo...
  → 1249 registros
  → Columnas: ['c_digo_dane_departamento', 'nombre', 'c_digo_dane_municipio', 'nombre_1', 'tipo_municipio_isla_rea_no', 'categoria_municipio', 'comisarias_ley_2126_100_000', 'existencia_de_comisarias', 'nombre_comisaria', 'direcci_n_comisara', 'telefono_de_contacto', 'horario_de_atenci_n', 'longitud_de_ubicaci_n_de', 'latitud_de_ubicaci_n_de_la', 'coordenadas_de_ubicaci_n']
    🔄 'nombre' → 'departamento'
    🔄 [Comisarias] 'nombre_1' → 'municipio'
  ✅ Comisarias: 1248 registros
  💾 C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\data thinking\2segunda entrega\LexData\notebooks\03_data_judicial\lexdata_comisarias_directorio.csv (1,248 filas)


,c_digo_dane_departamento,departamento,c_digo_dane_municipio,municipio,tipo_municipio_isla_rea_no,categoria_municipio,comisarias_ley_2126_100_000,existencia_de_comisarias,nombre_comisaria,direcci_n_comisara,telefono_de_contacto,horario_de_atenci_n,longitud_de_ubicaci_n_de,latitud_de_ubicaci_n_de_la,coordenadas_de_ubicaci_n,fuente
0,5,ANTIOQUIA,5088,BELLO,Municipio,1,6,1,Comisaría Cuarta,Calle 20D # 42A-65,Null,lunes a jueves de 7:00 a. m. a 12:00 m. y de 1...,6307490,"-75,551,670","6°18'27.0""N 75°33'06.0""W",Directorio Comisarias Ley 2126
1,5,ANTIOQUIA,5266,ENVIGADO,Municipio,1,3,2,Comisaría de Familia Tercera,Calle 40 Sur# 24F106,604) 3394000 ext. 4035,Lunes a Viernes de\n7:00 a.m. a 12:00 m y\nde ...,6148884,"-75,577,145","6°08'56.0""N 75°34'37.7""W",Directorio Comisarias Ley 2126
2,5,ANTIOQUIA,5266,ENVIGADO,Municipio,1,3,2,Comisaría de Familia Cuarta,Calle 40 Sur# 24F106,604) 3394000 ext. 4091,Lunes a Viernes de\n7:00 a.m. a 12:00 m y\nde ...,6148884,"-75,577,145","6°08'56.0""N 75°34'37.7""W",Directorio Comisarias Ley 2126


### Fuente 6 — CSJ Rama Judicial — Procesos de Alimentos *(NUEVO v9)*

In [9]:
# Reemplaza el proxy 0.75xVIF con datos reales de la Rama Judicial
CSJ_IDS_A_PROBAR = ["x5yx-c7vy"]


def scrape_csj_alimentos(session, years: list) -> pd.DataFrame:
    """
    Rama Judicial (CSJ) — Procesos de alimentos.
    v9: fuente real que reemplaza proxy alimentos = 0.75 x VIF.
    Si el ID principal falla, busca en catalogo.
    """
    DID = None
    col_depto_det = None

    for did in CSJ_IDS_A_PROBAR:
        info = inspect_dataset(session, did)
        if info["disponible"] and info["columnas"]:
            DID = did
            col_depto_det = detectar_col_depto(info["columnas"])
            print(f"  ✅ CSJ Alimentos: ID {did} disponible")
            print(f"     Columnas: {info['columnas'][:10]}")
            break
        else:
            print(f"  ✗ CSJ ID {did}: {info.get('error', 'no disponible')}")

    if not DID:
        print("  → Buscando en catalogo Socrata...")
        url = "https://api.us.socrata.com/api/catalog/v1"
        queries_csj = [
            "procesos alimentos familia rama judicial colombia",
            "demandas alimentos juzgado familia colombia",
            "expedientes judiciales alimentos colombia",
        ]
        for q in queries_csj:
            try:
                r = session.get(url,
                    params={"q": q, "domains": "www.datos.gov.co", "limit": 8},
                    timeout=12)
                for ds in r.json().get("results", []):
                    did_c = ds["resource"]["id"]
                    nombre_c = ds["resource"]["name"]
                    info_c = inspect_dataset(session, did_c)
                    if info_c["disponible"] and info_c["columnas"]:
                        DID = did_c
                        col_depto_det = detectar_col_depto(info_c["columnas"])
                        print(f"  ✅ Catalogo CSJ: '{nombre_c}' | {did_c}")
                        break
                if DID:
                    break
            except Exception as e:
                print(f"  ⚠ Error catalogo: {e}")

    if not DID:
        print("  ✗ CSJ Alimentos: no encontrado")
        print("    → alimentos_familia_total usara proxy temporal (0.75 x VIF)")
        return pd.DataFrame()

    print(f"\n[CSJ Alimentos] Extrayendo — ID: {DID}...")
    params = {}
    if col_depto_det and DEPARTAMENTOS_FILTRO:
        params["$where"] = build_where_depto(col_depto_det)

    raw = socrata_get(session, DID, params, max_pages=300)
    if not raw:
        raw = socrata_get(session, DID, {}, max_pages=300)
    print(f"  → {len(raw)} registros")

    if not raw:
        return pd.DataFrame()

    df = pd.DataFrame(raw).drop_duplicates()
    print(f"  → Columnas: {list(df.columns)}")
    df = normalizar_columnas(df, fuente_log="CSJ")

    if "departamento" in df.columns:
        df["departamento"] = df["departamento"].apply(normalizar_texto)
        df = filtrar_depto(df)

    df["municipio"] = df["municipio"].apply(normalizar_texto) \
                      if "municipio" in df.columns else "SIN_DATO"

    if "anio" not in df.columns:
        col_fecha = next(
            (c for c in ["fecha_radicacion", "fecha_hecho", "fecha"] if c in df.columns),
            None
        )
        if col_fecha:
            df["anio"] = pd.to_datetime(df[col_fecha], dayfirst=True,
                                         errors="coerce").dt.year.astype("Int64")
        else:
            col_a = detectar_col_fecha(df)
            if col_a:
                df = df.rename(columns={col_a: "anio"})
    if "anio" in df.columns:
        df = filtrar_anios(df, "anio")

    df["tipo_ciclo"] = "ALIMENTOS"
    df["fuente"] = f"CSJ — Procesos Alimentos ({DID})"
    df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(1) \
                     if "cantidad" in df.columns else 1

    print(f"  ✅ CSJ Alimentos: {len(df):,} registros · {df['municipio'].nunique()} municipios")
    return df


df_csj_alimentos = scrape_csj_alimentos(session_main, YEARS)
safe_csv(df_csj_alimentos, "lexdata_csj_alimentos.csv")
df_csj_alimentos.head(3)

  ✗ CSJ ID x5yx-c7vy: HTTP 404
  → Buscando en catalogo Socrata...
  ✅ Catalogo CSJ: 'Tienda Virtual del Estado Colombiano - Consolidado' | rgxm-mmea

[CSJ Alimentos] Extrayendo — ID: rgxm-mmea...
  → 163853 registros
  → Columnas: ['a_o', 'identificador_de_la_orden', 'rama_de_la_entidad', 'orden_de_la_entidad', 'sector_de_la_entidad', 'entidad', 'solicitante', 'fecha', 'fecha_vence', 'proveedor', 'estado', 'solicitud', 'items', 'total', 'agregacion', 'ciudad', 'entidad_obigada', 'espostconflicto', 'nit_proveedor', 'actividad_economica_proveedor', 'nit_entidad', 'id_entidad']
    🔄 [CSJ] 'a_o' → 'anio'
    📅 Filtro anios 2020–2025: 163853 → 114650 registros
  ✅ CSJ Alimentos: 114,650 registros · 1 municipios
  💾 C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\data thinking\2segunda entrega\LexData\notebooks\03_data_judicial\lexdata_csj_alimentos.csv (114,650 filas)


,anio,identificador_de_la_orden,rama_de_la_entidad,orden_de_la_entidad,sector_de_la_entidad,entidad,solicitante,fecha,fecha_vence,proveedor,...,entidad_obigada,espostconflicto,nit_proveedor,actividad_economica_proveedor,nit_entidad,id_entidad,municipio,tipo_ciclo,fuente,cantidad
0,2024,127570,No Definido,No Definido,No Definido,CARCEL Y PENITENCIARIA DE MEDIA SEGURIDAD DE G...,LINA PAOLA LADINO TORRES,2024-04-19T00:00:00.000,2024-05-20T00:00:00.000,POLYFLEX,...,No Definido,No Definido,No Aplica,No Definido,No Definido,1209,SIN_DATO,ALIMENTOS,CSJ — Procesos Alimentos (rgxm-mmea),1
1,2025,146373,No Definido,No Definido,No Definido,CARCEL Y PENITENCIARIA DE MEDIA SEGURIDAD DE B...,LUANA SOFIA GARCIA MOSQUERA,2025-05-20T00:00:00.000,2025-07-30T00:00:00.000,POLYFLEX Y/O JAIME BELTR+ÜN URIBE,...,No Definido,No Definido,No Aplica,No Definido,No Definido,1116,SIN_DATO,ALIMENTOS,CSJ — Procesos Alimentos (rgxm-mmea),1
3,2020,55588,No Definido,No Definido,No Definido,CARCEL Y PENITENCIARIA DE MEDIA SEGURIDAD DE T...,GONZALO RIVERA DUQUE,2020-09-23T00:00:00.000,2020-12-15T00:00:00.000,UNION TEMPORAL LA RECETTA - NUTRESA,...,No Definido,No Definido,No Aplica,No Definido,No Definido,1531,SIN_DATO,ALIMENTOS,CSJ — Procesos Alimentos (rgxm-mmea),1


## Sección 5 — Construcción del IVF

In [10]:
def agregar_por_municipio_anio(df: pd.DataFrame, tipo: str) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=["municipio", "anio", f"{tipo}_total"])
    df = df.copy()
    if "anio" not in df.columns:
        df["anio"] = 9999
    if "cantidad" in df.columns:
        df["cantidad"] = pd.to_numeric(df["cantidad"], errors="coerce").fillna(0)
        agg = df.groupby(["municipio", "anio"], as_index=False)["cantidad"].sum()
        agg = agg.rename(columns={"cantidad": f"{tipo}_total"})
    else:
        agg = df.groupby(["municipio", "anio"], as_index=False).size()
        agg = agg.rename(columns={"size": f"{tipo}_total"})
    return agg


agg_vif_inmlcf   = agregar_por_municipio_anio(df_vif_inmlcf,    "vif_inmlcf")
agg_vif_policia  = agregar_por_municipio_anio(df_vif_policia,   "vif_policia")
agg_icbf         = agregar_por_municipio_anio(df_icbf,          "medidas_proteccion")
agg_inasistencia = agregar_por_municipio_anio(df_inasistencia,  "inasistencia")
agg_csj_alim     = agregar_por_municipio_anio(df_csj_alimentos, "alimentos_familia")

print(f"  agg_vif_inmlcf:    {len(agg_vif_inmlcf)} filas")
print(f"  agg_vif_policia:   {len(agg_vif_policia)} filas")
print(f"  agg_icbf:          {len(agg_icbf)} filas")
print(f"  agg_inasistencia:  {len(agg_inasistencia)} filas")
print(f"  agg_csj_alimentos: {len(agg_csj_alim)} filas  <- NUEVO v9")

# Consolidar VIF (INMLCF + Policia)
dfs_vif = [d for d in [agg_vif_inmlcf, agg_vif_policia] if not d.empty]
if dfs_vif:
    df_vif_total = pd.concat(dfs_vif, join="outer") \
                     .groupby(["municipio", "anio"], as_index=False) \
                     .sum(numeric_only=True)
    cols_vif = [c for c in ["vif_inmlcf_total", "vif_policia_total"] if c in df_vif_total.columns]
    df_vif_total["vif_total"] = df_vif_total[cols_vif].sum(axis=1)
else:
    df_vif_total = pd.DataFrame(columns=["municipio", "anio", "vif_total"])

# Feature matrix — outer join progresivo
dfs_merge = [df_vif_total[["municipio", "anio", "vif_total"]]]
for agg_df in [agg_inasistencia, agg_icbf, agg_csj_alim]:
    if not agg_df.empty:
        dfs_merge.append(agg_df)

df_matrix = dfs_merge[0]
for d in dfs_merge[1:]:
    df_matrix = df_matrix.merge(d, on=["municipio", "anio"], how="outer")

# Alimentos: datos reales si CSJ disponible; proxy solo como ultimo recurso
if "alimentos_familia_total" not in df_matrix.columns:
    df_matrix["alimentos_familia_total"] = (
        pd.to_numeric(df_matrix.get("vif_total", 0), errors="coerce").fillna(0) * 0.75
    ).round()
    print("⚠ alimentos_familia_total: PROXY temporal (0.75 x VIF)")
    print("  → CSJ x5yx-c7vy no disponible. Verificar en datos.gov.co")
else:
    print("✅ alimentos_familia_total: datos reales CSJ")

for col in ["vif_total", "alimentos_familia_total", "medidas_proteccion_total", "inasistencia_total"]:
    if col not in df_matrix.columns:
        df_matrix[col] = 0
    df_matrix[col] = pd.to_numeric(df_matrix[col], errors="coerce").fillna(0)

print(f"\n✅ Feature matrix: {df_matrix.shape[0]} filas · {df_matrix.shape[1]} columnas")
print(f"   Municipios: {df_matrix['municipio'].nunique()}")
print(f"   Anios: {sorted(df_matrix['anio'].dropna().unique().tolist())}")
df_matrix.head()

  agg_vif_inmlcf:    3629 filas
  agg_vif_policia:   5603 filas
  agg_icbf:          55 filas
  agg_inasistencia:  55 filas
  agg_csj_alimentos: 6 filas  <- NUEVO v9
✅ alimentos_familia_total: datos reales CSJ

✅ Feature matrix: 6018 filas · 6 columnas
   Municipios: 1062
   Anios: [2020, 2021, 2022, 2023, 2024, 2025, 9999]


,municipio,anio,vif_total,inasistencia_total,medidas_proteccion_total,alimentos_familia_total
0,,2023,2.0,0.0,0.0,0.0
1,,2024,1.0,0.0,0.0,0.0
2,,2025,1.0,0.0,0.0,0.0
3,ABEJORRAL,2020,5.0,0.0,0.0,0.0
4,ABEJORRAL,2021,18.0,0.0,0.0,0.0


In [11]:
def calcular_ivf(df: pd.DataFrame, pesos: dict, pob_dict: dict) -> pd.DataFrame:
    df = df.copy()
    df["ivf_score_bruto"] = sum(
        df[col].fillna(0) * peso
        for col, peso in pesos.items()
        if col in df.columns
    )
    min_s, max_s = df["ivf_score_bruto"].min(), df["ivf_score_bruto"].max()
    if max_s > min_s:
        df["ivf_score_ponderado"] = (
            (df["ivf_score_bruto"] - min_s) / (max_s - min_s) * 100
        ).round(1)
    else:
        df["ivf_score_ponderado"] = 50.0
    df["poblacion"] = df["municipio"].map(pob_dict).fillna(50000)
    df["ivf_tasa_100k"] = (df["ivf_score_bruto"] / df["poblacion"] * 100_000).round(2)
    p33 = df["ivf_score_ponderado"].quantile(0.33)
    p66 = df["ivf_score_ponderado"].quantile(0.66)
    df["nivel_riesgo"] = pd.cut(
        df["ivf_score_ponderado"],
        bins=[-1, p33, p66, 101],
        labels=["BAJO", "MEDIO", "ALTO"]
    )
    return df


df_ivf = calcular_ivf(df_matrix, PESOS_IVF, DANE_POB_2024)

agg_cols = {c: "sum" for c in ["vif_total", "alimentos_familia_total",
                                 "medidas_proteccion_total", "inasistencia_total",
                                 "ivf_score_bruto"]}
agg_cols.update({"ivf_score_ponderado": "mean", "ivf_tasa_100k": "mean"})
agg_cols_presentes = {k: v for k, v in agg_cols.items() if k in df_ivf.columns}

df_ivf_resumen = (
    df_ivf.groupby("municipio", as_index=False)
    .agg(agg_cols_presentes)
    .sort_values("ivf_score_ponderado", ascending=False)
    .reset_index(drop=True)
)
p75 = df_ivf_resumen["ivf_score_ponderado"].quantile(0.75)
df_ivf_resumen["alerta"] = df_ivf_resumen["ivf_score_ponderado"] >= p75

print(f"✅ IVF calculado — {len(df_ivf_resumen)} municipios")
print(f"   Umbral alerta (P75): {p75:.1f}")
print(f"   Municipios en alerta: {df_ivf_resumen['alerta'].sum()}")

cols_show = [c for c in ["municipio", "vif_total", "alimentos_familia_total",
                          "inasistencia_total", "ivf_score_ponderado",
                          "ivf_tasa_100k", "alerta"]
             if c in df_ivf_resumen.columns]
print(df_ivf_resumen[cols_show].head(12).to_string(index=False))

✅ IVF calculado — 1062 municipios
   Umbral alerta (P75): 0.2
   Municipios en alerta: 275
        municipio  vif_total  alimentos_familia_total  inasistencia_total  ivf_score_ponderado  ivf_tasa_100k  alerta
         SIN_DATO        0.0                 114650.0                 0.0            91.916667   11465.000000    True
     BOGOTA, D.C.    25842.0                      0.0                 0.0            33.140000    4134.720000    True
 BOGOTA D.C. (CT)    11658.0                      0.0                 0.0            12.483333    1554.400000    True
         MEDELLIN     6398.0                      0.0                 0.0             8.200000    1023.680000    True
           SOACHA     6651.0                      0.0                 0.0             7.100000     886.800000    True
    MEDELLIN (CT)     6069.0                      0.0                 0.0             6.500000     809.200000    True
        CALI (CT)     4797.0                      0.0                 0.0          

## Sección 6 — Exportación (CSV + script SQL para PostgreSQL)

In [12]:
safe_csv(df_ivf,         "lexdata_co_ocurrencia_IVF_v9.csv")
safe_csv(df_ivf_resumen, "lexdata_ivf_resumen_municipios_v9.csv")

SQL_SCHEMA = '''
-- LexData v9 — Schema PostgreSQL
-- Ejecutar: psql -U postgres -d lexdata -f lexdata_schema_v9.sql

CREATE SCHEMA IF NOT EXISTS lexdata;

CREATE TABLE IF NOT EXISTS lexdata.ivf_municipios (
    id                        SERIAL PRIMARY KEY,
    municipio                 TEXT NOT NULL,
    anio                      INTEGER,
    vif_total                 NUMERIC DEFAULT 0,
    alimentos_familia_total   NUMERIC DEFAULT 0,
    medidas_proteccion_total  NUMERIC DEFAULT 0,
    inasistencia_total        NUMERIC DEFAULT 0,
    ivf_score_bruto           NUMERIC,
    ivf_score_ponderado       NUMERIC,
    ivf_tasa_100k             NUMERIC,
    nivel_riesgo              TEXT,
    poblacion                 INTEGER,
    alerta                    BOOLEAN DEFAULT FALSE,
    created_at                TIMESTAMP DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS lexdata.hechos_vif (
    id           SERIAL PRIMARY KEY,
    municipio    TEXT,
    departamento TEXT,
    anio         INTEGER,
    cantidad     NUMERIC DEFAULT 1,
    tipo_ciclo   TEXT DEFAULT\'VIF\',
    fuente       TEXT,
    created_at   TIMESTAMP DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS lexdata.inasistencia_alimentaria (
    id           SERIAL PRIMARY KEY,
    municipio    TEXT,
    departamento TEXT,
    anio         INTEGER,
    cantidad     NUMERIC DEFAULT 1,
    tipo_ciclo   TEXT DEFAULT\'INASISTENCIA\',
    fuente       TEXT,
    created_at   TIMESTAMP DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS lexdata.medidas_icbf (
    id           SERIAL PRIMARY KEY,
    municipio    TEXT,
    departamento TEXT,
    anio         INTEGER,
    cantidad     NUMERIC DEFAULT 1,
    tipo_ciclo   TEXT DEFAULT\'MEDIDAS_PROTECCION\',
    fuente       TEXT,
    created_at   TIMESTAMP DEFAULT NOW()
);

CREATE TABLE IF NOT EXISTS lexdata.procesos_alimentos (
    id           SERIAL PRIMARY KEY,
    municipio    TEXT,
    departamento TEXT,
    anio         INTEGER,
    cantidad     NUMERIC DEFAULT 1,
    tipo_ciclo   TEXT DEFAULT\'ALIMENTOS\',
    fuente       TEXT,
    created_at   TIMESTAMP DEFAULT NOW()
);

CREATE INDEX IF NOT EXISTS idx_ivf_municipio ON lexdata.ivf_municipios(municipio);
CREATE INDEX IF NOT EXISTS idx_ivf_anio      ON lexdata.ivf_municipios(anio);
CREATE INDEX IF NOT EXISTS idx_ivf_alerta    ON lexdata.ivf_municipios(alerta);
'''

ruta_sql = os.path.join(OUTPUT_DIR, "lexdata_schema_v9.sql")
with open(ruta_sql, "w", encoding="utf-8") as f:
    f.write(SQL_SCHEMA)
print(f"💾 Schema SQL → {ruta_sql}")

  💾 C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\data thinking\2segunda entrega\LexData\notebooks\03_data_judicial\lexdata_co_ocurrencia_IVF_v9.csv (6,018 filas)
  💾 C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\data thinking\2segunda entrega\LexData\notebooks\03_data_judicial\lexdata_ivf_resumen_municipios_v9.csv (1,062 filas)
💾 Schema SQL → C:\Users\Acer\OneDrive\Escritorio\CARPETAS\septimo semestre\data thinking\2segunda entrega\LexData\notebooks\03_data_judicial\lexdata_schema_v9.sql


In [13]:
print()
print("=" * 65)
print("REPORTE DE COBERTURA — Pipeline v9")
print("=" * 65)

fuentes_info = {
    "INMLCF VIF":            (df_vif_inmlcf,    "VIF",          0.40),
    "Policia SIEDCO VIF":    (df_vif_policia,   "VIF",          0.40),
    "Inasistencia alim.": (df_inasistencia,  "INASISTENCIA", 0.10),
    "ICBF Medidas":          (df_icbf,          "MEDIDAS_ICBF", 0.20),
    "CSJ Alimentos (real)":  (df_csj_alimentos, "ALIMENTOS",    0.30),
    "Comisarias dir.":        (df_comisarias,    "DIRECTORIO",   None),
}

for nombre, (df_f, dim, peso) in fuentes_info.items():
    if df_f.empty:
        estado = "⚠ VACIO"
        detalle = ""
    else:
        muns = df_f["municipio"].nunique() if "municipio" in df_f.columns else "N/A"
        anios_f = sorted(df_f["anio"].dropna().unique().tolist()) \
                  if "anio" in df_f.columns else []
        estado = f"{len(df_f):>8,} filas"
        detalle = f"· {muns} municipios · {anios_f[:3]}{'...' if len(anios_f) > 3 else ''}"
    peso_str = f"(IVF peso: {peso})" if peso else ""
    print(f"  {'✅' if not df_f.empty else '⚠'} {nombre:<25} {estado}  {detalle}  {peso_str}")

print("-" * 65)
alimentos_fuente = "CSJ REAL" if not df_csj_alimentos.empty else "PROXY 0.75xVIF ⚠"
print(f"  Fuente alimentos_familia_total: {alimentos_fuente}")
print(f"  Feature matrix IVF: {len(df_ivf):,} filas · {df_ivf['municipio'].nunique()} municipios")
print(f"  Municipios en alerta (P75): {df_ivf_resumen['alerta'].sum()}")
print("=" * 65)
print()
print("Archivos generados en data_judicial/:")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    ruta_f = os.path.join(OUTPUT_DIR, fname)
    kb = os.path.getsize(ruta_f) / 1024
    print(f"  📄 {fname:<50} {kb:>7.1f} KB")


REPORTE DE COBERTURA — Pipeline v9
  ✅ INMLCF VIF                  98,922 filas  · 947 municipios · [2020, 2021, 2022]...  (IVF peso: 0.4)
  ✅ Policia SIEDCO VIF         202,965 filas  · 1016 municipios · [2020, 2021, 2022]...  (IVF peso: 0.4)
  ✅ Inasistencia alim.              56 filas  · 55 municipios · []  (IVF peso: 0.1)
  ✅ ICBF Medidas                    56 filas  · 55 municipios · []  (IVF peso: 0.2)
  ✅ CSJ Alimentos (real)       114,650 filas  · 1 municipios · [2020, 2021, 2022]...  (IVF peso: 0.3)
  ✅ Comisarias dir.              1,248 filas  · 1020 municipios · []  
-----------------------------------------------------------------
  Fuente alimentos_familia_total: CSJ REAL
  Feature matrix IVF: 6,018 filas · 1062 municipios
  Municipios en alerta (P75): 275

Archivos generados en data_judicial/:
  📄 lexdata_alertas_tempranas.csv                        106.9 KB
  📄 lexdata_co_ocurrencia_IVF_v9.csv                     317.3 KB
  📄 lexdata_comisarias_directorio.csv           

## Notas técnicas v9

| Problema v8 | Solución v9 |
|---|---|
| `alimentos_familia_total` era proxy `0.75×VIF` | Fuente real CSJ `x5yx-c7vy` + busqueda catalogo |
| ICBF fallback aceptaba Susa (1 municipio Cundinamarca) | Verificacion cobertura: exige >=10 departamentos o Valle del Cauca |
| Inasistencia: solo 1 ID | `hf4m-4hbq` + `gthr-pj5f` + catalogo automatico |
| CSVs sin tipos limpios | `safe_csv()`: float→int + UTF-8 BOM |
| Sin schema Postgres | `lexdata_schema_v9.sql` con tablas, indices y COPY |

**IVF v9 — fuentes:**
| Dimension | Peso | Fuente | Tipo |
|---|---|---|---|
| VIF | 0.40 | INMLCF + Policia SIEDCO | Delito/lesion directa |
| Alimentos familia | 0.30 | **CSJ Rama Judicial (real)** | Civil — ruptura economica |
| Medidas ICBF | 0.20 | ICBF comisarias | Intervencion institucional |
| Inasistencia alimentaria | 0.10 | Fiscalia SPOA | Penal — incumplimiento familiar |

**Siguiente paso:** ejecutar el script SQL para cargar a PostgreSQL, luego reentrenar el modelo con datos reales.